In [ ]:
from preprocessing import RTL_cleanup

dir = str(input("Path"))
df_filtered, pop_dfs = RTL_cleanup(dir)

In [ ]:
from plotting_methods import RTL_boxplot

for key in pop_dfs.keys():
    df = pop_dfs[key]
    if df.Tissue.nunique() < 2:
        print(f"Skipping {key} due to insufficient tissue types.")
        continue

    RTL_boxplot(df, key)

In [ ]:
from plotting_methods import population_boxplot

population_boxplot(df_filtered, split='Population', annotate=True, subset=None)

In [ ]:
population_boxplot(df_filtered, split='Age group', annotate=True,
                   subset=['CD103+CD69+ TEM CD8'])

In [ ]:
df_test = df_filtered[df_filtered['Population'].isin(
    ['CD103+CD69+ TEM CD8', 'CD103-CD69- TEM CD8'])]

population_boxplot(df_test, split='Population', annotate=True, subset=None)

In [ ]:
from plotting_methods import population_boxplot

population_boxplot(df_filtered, split='Age group',
                   annotate=True, subset=['CD103+CD69+ TEM CD8'], plot_donors=True)

In [ ]:
df_test = df_filtered[df_filtered['Tissue'].isin(['LNG'])]
population_boxplot(df_test, split='Tissue residence', annotate=True)

In [ ]:
from plotting_methods import age_RTL_plot

for key in pop_dfs.keys():
    df = pop_dfs[key].sort_values('Age')
    if df.Tissue.nunique() < 2:
        print(f"Skipping {key} due to insufficient tissue types.")
        continue
    age_RTL_plot(df, key)

In [ ]:
from regression_analysis import regression_pipeline

reg_model, importance_df, clean_names = regression_pipeline(
    df_filtered, target='RTL', features=['Age', 'Tissue', 'Population'])

In [ ]:
from sklearn.tree import export_text

tree_rules = export_text(
    reg_model.named_steps['regressor'].estimators_[0],
    feature_names=clean_names,
    max_depth=3
)
print(tree_rules)

In [ ]:
import matplotlib.cm as cm
import matplotlib.pyplot as plt
from sklearn.tree import plot_tree

rf_step = reg_model.named_steps['regressor']

plt.figure(figsize=(24, 10), dpi=300)

with plt.rc_context({'image.cmap': 'coolwarm'}):
    plot_tree(
        rf_step.estimators_[0],
        max_depth=2,
        feature_names=clean_names,
        filled=True,
        rounded=True,
        precision=5,
        fontsize=11
    )

plt.title("Decision Tree Path for Relative Telomere Length (RTL) Prediction",
          fontsize=16, fontweight='bold', pad=20)
plt.show()
plt.show()

In [ ]:
from regression_analysis import clean_encoder_map
feature_map = clean_encoder_map(reg_model)